In [9]:
import duckdb
import pandas as pd
import numpy as np
from datetime import datetime

# today
today = datetime.today().strftime("%y%m%d")

## Pull publications for manual review

In [17]:
# Connect to database
db = duckdb.connect("publications.db")
db.execute("PRAGMA memory_limit='3GB'")
db.execute("PRAGMA temp_directory='C:/temp/duckdb_spill'")
db.execute("PRAGMA threads=2")

In [18]:
# Show tables in database
db.sql("SHOW TABLES")

┌─────────────────────────┐
│          name           │
│         varchar         │
├─────────────────────────┤
│ publications_classified │
│ publications_embedding  │
│ run_260708_0000         │
│ run_260708_0000_reverse │
│ run_260708_1307         │
│ run_260708_1307_reverse │
└─────────────────────────┘

#### Get data from publications_classified

In [25]:
# Create dataframe from classified publications
data = db.sql("SELECT * FROM publications_classified").df()

In [26]:
data = data[data['date_dimensions'] == "training"]

In [21]:
# Filter data for publications that went through LLM scoping
data = data[data['pred_combined'] == 'in']

In [ ]:
# If this this is a review on an updated dataset, meaning hte columns scope_curated and pillar_curated already exist, perform this filtering step
# data = data[data['scope_curated'].isna()]

#### Create filter mask for curated scope and pillar 

In [22]:
# decision mask for manual review
inherit_mask = (
    # scope_LLM=in, confident, and a pillar was assigned
    ((data['scope_LLM'] == 'in') & (data['confidence_LLM'] > 4) & (data['pillar_LLM'] != 'NA')) | 

    # scope_LLM=in, borderline confidence, but ML agrees on the pillar
    ((data['scope_LLM'] == 'in') & (data['confidence_LLM'] == 4) & (data['pillar_LLM'] != 'NA') & (data['pillar_LLM'] == data['pred_pillar'])) | 

    # scope_LLM=out, confident, and no pillar was assigned
    ((data['scope_LLM'] == 'out') & (data['confidence_LLM'] < 2) & (data['pillar_LLM'] == 'NA'))
)

#### Assign curated scope/pillar or manual review

In [23]:
# Create curated scope and pillar columns with decision criteria
data['scope_curated']  = np.where(inherit_mask, data['scope_LLM'],  'manual_review')
data['pillar_curated'] = np.where(inherit_mask, data['pillar_LLM'], 'manual_review')
data['date_review'] = today

In [24]:
# How many publications have to go through manual review?
data['scope_curated'].value_counts()

scope_curated
in               356
manual_review     48
out               18
Name: count, dtype: int64

#### Save borderline publications for manual review

In [9]:
# Export publications for review
review = data[data['scope_curated'] == 'manual_review']
review.to_csv(f"data/publications_for_review_{today}.csv")

#### Save (automatically) curated scope and pillar back to publications_classified

In [10]:
# filter for the publications with clear scope and pillar assigment
data = data[(data['scope_curated'] != 'manual_review') & (~data['scope_curated'].isna())]

In [11]:
# create columns in publications_classified if they don't already exist
db.sql(f"ALTER TABLE publications_classified ADD COLUMN IF NOT EXISTS scope_curated VARCHAR")
db.sql(f"ALTER TABLE publications_classified ADD COLUMN IF NOT EXISTS pillar_curated VARCHAR")

In [12]:
# add to database
db.register('data', data[['id', 'scope_curated', 'pillar_curated']])
db.sql(f"""
    UPDATE publications_classified
    SET scope_curated     = data.scope_curated,
        pillar_curated    = data.pillar_curated
    FROM data
    WHERE publications_classified.id = data.id
""")

In [28]:
db.close()

## Assign manually curated scope and pillar to respective publications

In [22]:
# Connect to database
db = duckdb.connect("publications_llmtesting.db")

In [23]:
# load manually reviewed data
reviewed_data = pd.read_csv("data/publications_reviewed_260706.csv")

#### Add scope and pillar back to original dataset, using publication ID

In [35]:
# add to database
db.register('reviewed_data', reviewed_data[['id', 'scope_curated', 'pillar_curated']])
db.sql(f"""
    UPDATE publications_classified
    SET scope_curated     = reviewed_data.scope_curated,
        pillar_curated    = reviewed_data.pillar_curated
    FROM reviewed_data
    WHERE publications_classified.id = reviewed_data.id
""")

In [36]:
data = db.sql("SELECT * FROM publications_classified").df()

In [37]:
data[~data['scope_curated'].isna()]

,id,title,abstract,authors,concepts_relevant,date,funder_countries,funders,open_access,research_org_cities,...,confidence_LLM,pillar_LLM,plant_based_LLM,fermentation_LLM,cultivated_LLM,cross_cutting_LLM,status_LLM,stop_reason_LLM,scope_curated,pillar_curated
0,pub.1188588536,Application of Plant‐Based Proteins in the Dev...,Consumer preferences are shifting from traditi...,"[{""affiliations"":[{""city"":""Foshan"",""city_id"":1...","[application of plant‐based proteins, plant-ba...",2025-05-13,"[{""id"":""CN"",""name"":""China""}]","[{'acronym': 'NSFC', 'acronyms': ['NSFC'], 'ci...",[closed],"[{""id"":""1809858"",""name"":""Guangzhou""},{""id"":""11...",...,7.0,PB,True,False,False,False,ok,tool_use,in,PB
1,pub.1185255386,Alternative Protein-Based Meat and Fish Analog...,This study aimed to explore the extent of rese...,"[{""affiliations"":[{""city"":""Braga"",""city_id"":27...","[meat, Fish Analog, fish, alternative proteins...",2025-02-04,"[{""id"":""PT"",""name"":""Portugal""}]","[{'acronym': 'FCT', 'acronyms': ['FCT'], 'city...","[oa_all, gold]","[{""id"":""2742032"",""name"":""Braga""}]",...,7.0,CC,True,True,False,True,ok,tool_use,in,CC
2,pub.1187762379,Plant-Based Alternatives to Meat Products,Animal proteins have been used in the formulat...,"[{""affiliations"":[{""city"":""Newport"",""city_id"":...","[plant-based alternatives, meat products, meat...",2025-04-17,NaN,NaN,"[oa_all, gold]","[{""id"":""2653305"",""name"":""Chatham""},{""id"":""2641...",...,7.0,PB,True,False,False,False,ok,tool_use,in,PB
3,pub.1182803126,An Investigation of the Status of Commercial M...,"Meat analogs are a burgeoning industry, with p...","[{""affiliations"":[{""city"":""Seoul"",""city_id"":18...","[meat analogs, ingredients, meat, plant-based ...",2025-01-01,"[{""id"":""KR"",""name"":""South Korea""}]","[{'acronym': 'iPET', 'acronyms': ['iPET'], 'ci...","[oa_all, gold]","[{""id"":""1835848"",""name"":""Seoul""}]",...,6.0,CC,True,True,True,True,ok,tool_use,in,CC
4,pub.1194716889,A systematic review of consumer responses to m...,Meat alternatives have gained popularity. Howe...,"[{""affiliations"":[{""city"":""Seoul"",""city_id"":18...","[consumer responses, meat, meat alternatives, ...",2025-11-06,NaN,NaN,[closed],"[{""id"":""1835848"",""name"":""Seoul""}]",...,7.0,CC,True,False,True,True,ok,tool_use,in,CC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,pub.1195463920,Scaffolding Strategies in Mimicking Muscle and...,Cultivated meat is intended to solve the curre...,"[{""affiliations"": [{""city"": ""Dalian"", ""city_id...","[cultured meat, meat, cultivated meat, animal ...",2025-11-27,"[{""id"": ""CN"", ""name"": ""China""}]","[{'acronym': 'NSFC', 'acronyms': ['NSFC'], 'ci...",[closed],"[{""id"": ""1814087"", ""name"": ""Dalian""}]",...,7.0,CM,False,False,True,False,ok,tool_use,in,CM
96,pub.1184681018,Can cell-cultured meat from stem cells pave th...,"As the global population grows, the demand for...","[{""affiliations"": [{""city"": ""S\u00e3o Paulo"", ...","[cell-cultured meat, meat, alternative protein...",2025-01-21,"[{""id"": ""BR"", ""name"": ""Brazil""}, {""id"": ""US"", ...","[{'acronym': 'FAPESP', 'acronyms': ['FAPESP'],...","[oa_all, gold]","[{""id"": ""3448439"", ""name"": ""S\u00e3o Paulo""}]",...,7.0,CM,False,False,True,False,ok,tool_use,in,CM
97,pub.1185767741,Reassessing the sustainability promise of cult...,There are currently over 170 companies in the ...,"[{""affiliations"": [{""city"": ""Palmerston North""...","[cultured meat, meat, CM industry, animal welf...",2025-02-21,"""<NA>""",NaN,[closed],"[{""id"": ""2158177"", ""name"": ""Melbourne""}, {""id""...",...,7.0,CM,False,False,True,False,ok,tool_use,in,CM
98,pub.1200546129,Recent Advantage Of Oleogel Application In Foo...,The scientific community is confronting a nove...,"[{""affiliations"": [], ""corresponding"": """", ""cu...","[food, sensory properties, trans fat, fat, sat...",2025-07-17,"""<NA>""",NaN,"[oa_all, hybrid]","""<N

In [4]:
db.close()